# Advanced NLP Patterns & Combined Architectures

This notebook covers practical patterns and combined architectures for competition problems:
1. **Hybrid Seq2Seq-Transformer** models
2. **Attention weight visualization** techniques
3. **Transfer learning** patterns
4. **Custom loss functions** for NLP
5. **Inference optimizations**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =====================================================================
# ATTENTION WEIGHT VISUALIZATION
# =====================================================================

def visualize_attention_matrix(attention_weights, src_tokens=None, tgt_tokens=None):
    """
    Prepare attention weights for visualization.
    
    Args:
        attention_weights: [seq_len_tgt, seq_len_src] or [batch, seq_len_tgt, seq_len_src]
        src_tokens: List of source token strings
        tgt_tokens: List of target token strings
    
    Returns:
        dict with visualization data
    """
    if attention_weights.dim() == 3:
        attention_weights = attention_weights[0]  # First sample from batch
    
    return {
        'matrix': attention_weights.cpu().numpy(),
        'src_tokens': src_tokens,
        'tgt_tokens': tgt_tokens,
        'max_attention': attention_weights.max().item(),
        'min_attention': attention_weights.min().item(),
        'entropy': calculate_attention_entropy(attention_weights)
    }


def calculate_attention_entropy(attention_weights):
    """
    Calculate entropy of attention distribution.
    Lower entropy = more focused attention.
    """
    # Compute entropy for each target position
    entropy = -torch.sum(attention_weights * torch.log(attention_weights + 1e-10), dim=-1)
    return entropy.mean().item()


In [ ]:
# =====================================================================
# CUSTOM LOSS FUNCTIONS FOR NLP
# =====================================================================

class FocalLoss(nn.Module):
    """
    Focal Loss: Focuses on hard examples by down-weighting easy examples.
    Useful for imbalanced classification in NLP.
    
    L = -alpha * (1 - p_t)^gamma * log(p_t)
    """
    
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        p = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - p) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class LabelSmoothingLoss(nn.Module):
    """
    Label Smoothing: Prevents overconfident predictions.
    Replaces hard targets with soft targets: (1-eps)*y + eps/K
    """
    
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Create smoothed targets
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), self.confidence)
        
        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))


class ContrastiveLoss(nn.Module):
    """
    Contrastive Loss for Siamese networks.
    Pulls positive pairs together, pushes negative pairs apart.
    """
    
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, embed1, embed2, labels):
        # Euclidean distance
        distance = torch.norm(embed1 - embed2, p=2, dim=1)
        
        # Loss: pull similar pairs close, push dissimilar pairs apart
        loss = labels * distance.pow(2) + \
               (1 - labels) * F.relu(self.margin - distance).pow(2)
        
        return loss.mean()


class TripletLoss(nn.Module):
    """
    Triplet Loss: (anchor, positive, negative) tuples.
    Margin-based: d(a,p) + margin < d(a,n)
    """
    
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        pos_dist = torch.norm(anchor - positive, p=2, dim=1)
        neg_dist = torch.norm(anchor - negative, p=2, dim=1)
        
        loss = F.relu(pos_dist - neg_dist + self.margin)
        return loss.mean()


In [ ]:
# =====================================================================
# BEAM SEARCH FOR SEQUENCE GENERATION
# =====================================================================

class BeamSearchDecoder:
    """
    Beam search for decoding sequences.
    More efficient than exhaustive search, better than greedy decoding.
    """
    
    def __init__(self, beam_width=5, max_length=50, eos_token=2):
        self.beam_width = beam_width
        self.max_length = max_length
        self.eos_token = eos_token
    
    def decode(self, decoder_fn, initial_state, batch_size, device):
        """
        Args:
            decoder_fn: Function that takes (input_id, state) and returns (logits, new_state)
            initial_state: Initial decoder state
            batch_size: Batch size
            device: Torch device
        
        Returns:
            sequences: [batch_size, beam_width, max_length]
            scores: [batch_size, beam_width]
        """
        # Initialize with start token
        start_token = torch.tensor([[1]] * batch_size, device=device)  # SOS token
        
        # Expand for beam search
        sequences = start_token.unsqueeze(1).expand(batch_size, self.beam_width, -1)
        scores = torch.zeros(batch_size, self.beam_width, device=device)
        
        active_batch = batch_size
        
        for t in range(self.max_length - 1):
            # Get decoder outputs for current step
            current_tokens = sequences[:, :, -1].reshape(-1)
            logits, initial_state = decoder_fn(current_tokens, initial_state)
            
            logits = logits.reshape(batch_size, self.beam_width, -1)
            log_probs = F.log_softmax(logits, dim=-1)
            
            # Add previous scores
            scores_expanded = scores.unsqueeze(-1) + log_probs  # [batch, beam, vocab]
            
            # Get top-k
            flat_scores = scores_expanded.reshape(batch_size, -1)
            top_scores, top_indices = torch.topk(flat_scores, self.beam_width, dim=-1)
            
            # Update sequences and scores
            beam_indices = top_indices // logits.shape[-1]
            token_indices = top_indices % logits.shape[-1]
            
            # Reorder sequences
            sequences = torch.gather(sequences, 1, 
                                     beam_indices.unsqueeze(-1).expand(-1, -1, sequences.shape[-1]))
            
            # Append new tokens
            sequences = torch.cat([sequences, token_indices.unsqueeze(-1)], dim=-1)
            scores = top_scores
        
        return sequences, scores


In [ ]:
# =====================================================================
# INFERENCE OPTIMIZATION: QUANTIZATION & PRUNING
# =====================================================================

class QuantizedEmbedding(nn.Module):
    """
    Quantized embedding layer for reduced memory footprint.
    """
    
    def __init__(self, num_embeddings, embedding_dim, bits=8):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.bits = bits
        self.scale = nn.Parameter(torch.tensor(1.0))
    
    def quantize(self):
        """Quantize embeddings to lower precision"""
        weight = self.embedding.weight
        max_val = weight.abs().max()
        self.scale.data = max_val / (2 ** (self.bits - 1) - 1)
        
        # Quantize
        quantized = torch.round(weight / self.scale).clamp(-(2**(self.bits-1)), 2**(self.bits-1)-1)
        self.embedding.weight.data = quantized * self.scale
    
    def forward(self, x):
        return self.embedding(x)


def prune_weights(model, pruning_ratio=0.1):
    """
    Magnitude-based weight pruning.
    Sets smallest weights to zero.
    """
    for name, param in model.named_parameters():
        if 'weight' in name and len(param.shape) > 1:  # Not biases
            threshold = torch.kthvalue(
                param.abs().view(-1),
                int(param.numel() * pruning_ratio)
            ).values
            mask = param.abs() > threshold
            param.data = param.data * mask.float()
    
    return model


def knowledge_distillation_loss(teacher_logits, student_logits, temperature=4.0, alpha=0.7):
    """
    Knowledge Distillation: Train smaller student model using larger teacher.
    
    Loss = alpha * CE(student, hard_labels) + (1-alpha) * KL(student, teacher)
    """
    ce_loss = F.cross_entropy(student_logits, torch.argmax(teacher_logits, dim=1))
    
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    kl_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
    
    return alpha * ce_loss + (1 - alpha) * kl_loss * (temperature ** 2)


In [ ]:
# =====================================================================
# TRAINING UTILITIES
# =====================================================================

class GradientAccumulation:
    """
    Gradient accumulation for larger effective batch sizes.
    Useful when GPU memory is limited.
    """
    
    def __init__(self, model, accumulation_steps=4):
        self.model = model
        self.accumulation_steps = accumulation_steps
        self.step_count = 0
    
    def should_step(self):
        return (self.step_count + 1) % self.accumulation_steps == 0
    
    def backward(self, loss):
        # Scale loss by accumulation steps
        scaled_loss = loss / self.accumulation_steps
        scaled_loss.backward()
        self.step_count += 1
        return scaled_loss


def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """
    Learning rate scheduler: Linear warmup then linear decay.
    """
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return max(0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps)))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class EarlyStopping:
    """
    Early stopping to prevent overfitting.
    """
    
    def __init__(self, patience=3, verbose=False):
        self.patience = patience
        self.verbose = verbose
        self.best_loss = None
        self.counter = 0
    
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            if self.verbose:
                print(f"Validation loss improved to {val_loss:.4f}")
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print(f"Early stopping triggered after {self.counter} patient epochs")
                return True
        
        return False


In [ ]:
# =====================================================================
# EXAMPLE: COMPLETE TRAINING LOOP WITH ALL UTILITIES
# =====================================================================

def train_nlp_model(
    model, train_loader, val_loader, num_epochs=10, 
    learning_rate=1e-3, device='cpu', use_label_smoothing=True
):
    """
    Complete training loop with best practices for NLP.
    """
    # Loss function
    if use_label_smoothing:
        criterion = LabelSmoothingLoss(num_classes=model.output_dim, smoothing=0.1)
    else:
        criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    
    # Learning rate scheduler
    num_training_steps = len(train_loader) * num_epochs
    num_warmup_steps = int(0.1 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
    
    # Early stopping
    early_stopping = EarlyStopping(patience=3, verbose=True)
    
    # Gradient accumulation
    accumulator = GradientAccumulation(model, accumulation_steps=4)
    
    print("Starting training...\n")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Backward pass with accumulation
            scaled_loss = accumulator.backward(loss)
            train_loss += scaled_loss.item()
            
            # Optimizer step
            if accumulator.should_step():
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
        
        # Validation phase
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item()
        
        val_loss /= len(val_loader)
        train_loss /= len(train_loader)
        
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        # Early stopping check
        if early_stopping(val_loss, model):
            break
    
    print("\nTraining completed!")
    return model


In [ ]:
# =====================================================================
# DEMONSTRATION
# =====================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")
    
    # Test custom losses
    print("=" * 60)
    print("CUSTOM LOSS FUNCTIONS")
    print("=" * 60)
    
    batch_size = 16
    num_classes = 10
    
    logits = torch.randn(batch_size, num_classes)
    targets = torch.randint(0, num_classes, (batch_size,))
    
    focal_loss = FocalLoss()
    ls_loss = LabelSmoothingLoss(num_classes)
    
    l_focal = focal_loss(logits, targets)
    l_smooth = ls_loss(logits, targets)
    
    print(f"Focal Loss: {l_focal.item():.4f}")
    print(f"Label Smoothing Loss: {l_smooth.item():.4f}")
    print(f"✓ Loss functions working correctly!\n")
    
    # Test attention visualization
    print("=" * 60)
    print("ATTENTION VISUALIZATION")
    print("=" * 60)
    
    attn_weights = F.softmax(torch.randn(10, 15), dim=-1)
    vis_data = visualize_attention_matrix(
        attn_weights,
        src_tokens=[f"src_{i}" for i in range(15)],
        tgt_tokens=[f"tgt_{i}" for i in range(10)]
    )
    
    print(f"Attention matrix shape: {vis_data['matrix'].shape}")
    print(f"Attention entropy: {vis_data['entropy']:.4f}")
    print(f"✓ Attention visualization ready!\n")
    
    # Test pruning
    print("=" * 60)
    print("WEIGHT PRUNING")
    print("=" * 60)
    
    class SimpleNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(100, 50)
            self.fc2 = nn.Linear(50, 10)
        
        def forward(self, x):
            return self.fc2(torch.relu(self.fc1(x)))
    
    model = SimpleNet()
    original_params = sum(p.numel() for p in model.parameters())
    
    model = prune_weights(model, pruning_ratio=0.3)
    pruned_zeros = sum((p == 0).sum().item() for p in model.parameters())
    
    print(f"Original parameters: {original_params}")
    print(f"Pruned zeros: {pruned_zeros}")
    print(f"✓ Pruning applied!\n")
    
    print("✅ All advanced patterns working!")
